# Loan Approval Prediction

**Goal:** predict whether a loan application will be approved, from details about the applicant.

This is a classification problem (approved or not approved).

**Pipeline:** load data → explore → clean → encode → split → train models → check results → which clues matter

Data: Kaggle "Loan Prediction Problem Dataset" (the training file, which has the loan status).

## Step 1: Load the data
Upload the train file first: click the folder icon on the left, then the upload icon, and wait until the file name appears in the list. (Colab deletes uploaded files when a session resets, so do this each time.) Then use the exact file name below.

In [1]:
import pandas as pd

df = pd.read_csv("train_u6lujuX_CVtuZ9i.csv")
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


## Step 2: Explore the data

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [3]:
print(df['Loan_Status'].value_counts())
print()
print(df['Loan_Status'].value_counts(normalize=True))

Loan_Status
Y    422
N    192
Name: count, dtype: int64

Loan_Status
Y    0.687296
N    0.312704
Name: proportion, dtype: float64


About 69% of the loans were approved (Y) and 31% were not (N). A model that always said "approved" would be about 69% accurate and would never reject anyone, so accuracy alone is not enough.

In [4]:
df['Dependents'].unique()

array(['0', '1', '2', '3+', nan], dtype=object)

`Dependents` has the value '3+', which is why the column is text and not numbers. Several columns have missing values (`nan`).

In [5]:
print(df.isnull().sum())

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


## Step 3: Clean the data
How many rows would be lost if we dropped every row that has a missing value?

In [6]:
print(df.shape)
print(df.dropna().shape)

(614, 13)
(480, 13)


Dropping would remove 134 of 614 rows (about 22%) from an already small dataset, so we fill the gaps instead: the most common value for text-like columns, and the median for `LoanAmount`. We also remove `Loan_ID`, which is only a label.

In [7]:
df = df.drop('Loan_ID', axis=1)

for col in ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History', 'Loan_Amount_Term']:
    df[col] = df[col].fillna(df[col].mode()[0])

df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

print(df.isnull().sum())

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64


## Step 4: Encode the text columns as numbers

In [8]:
# Dependents: '3+' becomes 3, then the column becomes numbers
df['Dependents'] = df['Dependents'].replace('3+', '3').astype(int)

# Answer column: Y = 1 (approved), N = 0 (not approved)
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

# Other text columns: one-hot encoding
df_encoded = pd.get_dummies(df, columns=['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area'])

print(df_encoded.shape)
df_encoded.head()

(614, 18)


,Dependents,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Gender_Female,Gender_Male,Married_No,Married_Yes,Education_Graduate,Education_Not Graduate,Self_Employed_No,Self_Employed_Yes,Property_Area_Rural,Property_Area_Semiurban,Property_Area_Urban
0,0,5849,0.0,128.0,360.0,1.0,1,False,True,True,False,True,False,True,False,False,False,True
1,1,4583,1508.0,128.0,360.0,1.0,0,False,True,False,True,True,False,True,False,True,False,False
2,0,3000,0.0,66.0,360.0,1.0,1,False,True,False,True,True,False,False,True,False,False,True
3,0,2583,2358.0,120.0,360.0,1.0,1,False,True,False,True,False,True,True,False,False,False,True
4,0,6000,0.0,141.0,360.0,1.0,1,False,True,True,False,True,False,True,False,False,False,True


## Step 5: Split into clues (X), answer (y), and train/test sets
`stratify=y` keeps the same share of approved and rejected loans in both sets. This matters because the dataset is small and the two groups are uneven.

In [9]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('Loan_Status', axis=1)   # clues
y = df_encoded['Loan_Status']                # the answer

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(491, 17)
(123, 17)
Loan_Status
1    0.686354
0    0.313646
Name: proportion, dtype: float64
Loan_Status
1    0.691057
0    0.308943
Name: proportion, dtype: float64


## Step 6: Model 1, Logistic Regression
Features are scaled first, because incomes are in thousands while columns like `Credit_History` are only 0 or 1.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, predictions))
print()
print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

Accuracy: 0.8536585365853658

[[22 16]
 [ 2 83]]

              precision    recall  f1-score   support

           0       0.92      0.58      0.71        38
           1       0.84      0.98      0.90        85

    accuracy                           0.85       123
   macro avg       0.88      0.78      0.81       123
weighted avg       0.86      0.85      0.84       123



The report is about the 123 test loans (38 that should be rejected, 85 that should be approved). The costly mistake for a lender is approving a loan that should have been rejected, so the number to watch is the recall of row `0`.

## Step 7: Model 2, balanced Logistic Regression
`class_weight='balanced'` makes mistakes on the smaller group (rejections) count for more.

In [11]:
lr_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_balanced.fit(X_train_scaled, y_train)
balanced_predictions = lr_balanced.predict(X_test_scaled)

print(confusion_matrix(y_test, balanced_predictions))
print()
print(classification_report(y_test, balanced_predictions))

[[26 12]
 [10 75]]

              precision    recall  f1-score   support

           0       0.72      0.68      0.70        38
           1       0.86      0.88      0.87        85

    accuracy                           0.82       123
   macro avg       0.79      0.78      0.79       123
weighted avg       0.82      0.82      0.82       123



The balanced model wrongly approves fewer bad loans (12 against 16), but it wrongly rejects more good loans (10 against 2). Which is better depends on what a bad loan costs the lender compared with a lost good customer. The test set is small, so treat the difference as a shift in balance, not a clear win.

## Step 8: What drives the decision?
Positive numbers push toward approval and negative numbers toward rejection. Pairs such as `Married_Yes` and `Married_No` show the same fact from two sides.

In [12]:
importance = pd.Series(lr_balanced.coef_[0], index=X.columns)
print(importance.sort_values(ascending=False))

Credit_History             1.228233
Property_Area_Semiurban    0.206854
Married_Yes                0.138901
Dependents                 0.092611
Education_Graduate         0.085749
Gender_Female              0.031631
Self_Employed_No           0.031377
Loan_Amount_Term           0.015315
ApplicantIncome            0.005294
Self_Employed_Yes         -0.031377
Gender_Male               -0.031631
LoanAmount                -0.076355
Property_Area_Urban       -0.076832
Education_Not Graduate    -0.085749
CoapplicantIncome         -0.133941
Married_No                -0.138901
Property_Area_Rural       -0.140301
dtype: float64


Check the main pattern directly in the data. Because `Loan_Status` is 1 for approved, the average is the approval rate.

In [13]:
print(df.groupby('Credit_History')['Loan_Status'].mean())

Credit_History
0.0    0.078652
1.0    0.790476
Name: Loan_Status, dtype: float64


## Key findings

- Applicants whose credit history meets the guidelines are approved 79.0% of the time, against 7.9% for applicants whose credit history does not.
- The model relied most on credit history, and the other clues mattered much less.

## Limitations

1. The dataset is small (614 loans, and only 123 in the test set), so the results could change with a different split.
2. Missing values were filled with guesses, for example the most common value for credit history, so the real approval rate for those applicants could be a bit different.
3. The model used details such as gender and marital status, and a real lender would have to think carefully about fairness before using them.